# Understading the data

In [18]:
import pandas as pd
import numpy as np

data = pd.read_csv("assets.csv")
data.head()

,Date,Asset_A,Asset_B,Asset_C,Asset_D,Asset_E,Asset_F
0,01-01-2021,97.631673,151.834946,81.354211,203.745418,119.902761,61.449194
1,04-01-2021,98.121243,151.888338,79.749046,204.263460,121.796180,61.313568
2,05-01-2021,96.641989,151.612706,81.664618,NaN,125.370824,59.737493
3,06-01-2021,95.975767,150.843503,80.797249,203.215367,126.889207,60.556546
4,07-01-2021,96.298141,150.929801,81.094938,204.805473,127.903711,58.966432


In [19]:
data = data.set_index("Date")
data.index = pd.to_datetime(data.index, dayfirst=True)
data.describe()

,Asset_A,Asset_B,Asset_C,Asset_D,Asset_E,Asset_F
count,720.000000,720.000000,717.000000,723.000000,712.000000,720.000000
mean,91.440599,167.114651,67.055719,189.432719,147.456798,76.770010
std,9.398711,16.078147,10.766879,23.557181,18.908105,10.130860
min,69.520009,131.309904,45.161825,130.762601,103.094623,56.475372
25%,83.660469,156.775838,60.758510,175.188570,133.312748,69.159265
50%,92.352385,167.784247,65.639441,193.336973,148.957368,76.330506
75%,99.184198,177.130434,71.660928,205.912551,159.823040,81.551437
max,109.815101,213.216224,99.259204,256.426550,189.848752,101.495826


## Cleaning data

### Finding the count missing values

In [21]:
def print_missing_values_count(data):
  for col in data.iloc(1):
    missing_val = col[col.isna()==True]
    print(f"{col.name}: {len(missing_val)}")

print_missing_values_count(data)

Asset_A: 21
Asset_B: 21
Asset_C: 24
Asset_D: 18
Asset_E: 29
Asset_F: 21


### Forward filling the missing values

In [22]:
cleaned_data = data.ffill(axis=0)

In [23]:
print_missing_values_count(cleaned_data)

Asset_A: 0
Asset_B: 0
Asset_C: 0
Asset_D: 0
Asset_E: 0
Asset_F: 0


In [27]:
import plotly.express as px

# Reset index so Date becomes a column
plot_data = cleaned_data.reset_index()

fig = px.line(plot_data, x='Date', y=plot_data.columns[1:],
              title="Assets Price Over Time")

fig.show()

## Calculating daily returns

In [25]:
# Returnt​=(Pricet​/Pricet−1​)−1
daily_returns = cleaned_data.copy()
daily_returns = cleaned_data.pct_change()
daily_returns = daily_returns[1:]

In [26]:
daily_returns.head()

,Asset_A,Asset_B,Asset_C,Asset_D,Asset_E,Asset_F
Date,,,,,,
2021-01-04,0.005014,0.000352,-0.019731,0.002543,0.015791,-0.002207
2021-01-05,-0.015076,-0.001815,0.024020,0.000000,0.029349,-0.025705
2021-01-06,-0.006894,-0.005073,-0.010621,-0.005131,0.012111,0.013711
2021-01-07,0.003359,0.000572,0.003684,0.007825,0.007995,-0.026258
2021-01-08,0.013333,0.013747,-0.001423,0.008600,0.003784,0.003353


In [ ]:
import plotly.graph_objects as go
import numpy as np

positive_colors = {
    "Asset_A": "#006400",
    "Asset_B": "#228B22",
    "Asset_C": "#2E8B57",
    "Asset_D": "#3CB371",
    "Asset_E": "#66CDAA",
    "Asset_F": "#98FB98"
}

negative_colors = {
    "Asset_A": "#8B0000",
    "Asset_B": "#B22222",
    "Asset_C": "#DC143C",
    "Asset_D": "#E9967A",
    "Asset_E": "#FA8072",
    "Asset_F": "#FFC0CB"
}

fig = go.Figure()

for asset in daily_returns.columns:

    asset_data = daily_returns[asset]

    positive = asset_data.where(asset_data >= 0)
    negative = asset_data.where(asset_data < 0)

    # Positive trace (shows in legend)
    fig.add_trace(
        go.Bar(
            x=daily_returns.index,
            y=positive,
            name=asset,
            marker_color=positive_colors[asset],
            legendgroup=asset,
            showlegend=True
        )
    )

    # Negative trace (hidden from legend but grouped)
    fig.add_trace(
        go.Bar(
            x=daily_returns.index,
            y=negative,
            marker_color=negative_colors[asset],
            legendgroup=asset,
            showlegend=False
        )
    )

fig.update_layout(
    title="Daily Returns – All Assets",
    xaxis_title="Date",
    yaxis_title="Daily Return",
    barmode="overlay",
    template="plotly_white",
    legend=dict(groupclick="togglegroup")  # Important line
)

fig.show()

In [34]:
def get_momentum_score(days: int, cleaned_data: pd.DataFrame):
  momentum_score_n_days = cleaned_data.copy()
  momentum_score_n_days = cleaned_data.pct_change(periods=days)
  momentum_score_n_days = momentum_score_n_days.iloc[days:]

  return momentum_score_n_days

In [71]:
momentum_score_30_days = get_momentum_score(days=30, cleaned_data=cleaned_data)
# momentum_score_30_days['Date'].astype('datetime64[ms]')

In [72]:
momentum_score_30_days.head()

,Asset_A,Asset_B,Asset_C,Asset_D,Asset_E,Asset_F
Date,,,,,,
2021-02-16,0.124790,-0.086215,-0.107976,0.019072,0.136644,-0.004703
2021-02-17,0.081113,-0.082016,-0.085806,0.035166,0.147102,-0.002501
2021-02-18,0.094754,-0.068583,-0.093296,0.030321,0.114395,0.011186
2021-02-19,0.113622,-0.072078,-0.083562,0.027153,0.101060,0.004235
2021-02-22,0.131928,-0.069524,-0.094988,0.003024,0.060297,0.004166


In [73]:
momentum_on_first_days = momentum_score_30_days.groupby(momentum_score_30_days.index.to_period("M")).first()
momentum_on_first_days = momentum_on_first_days[1:]

In [74]:
momentum_on_first_days.head()

,Asset_A,Asset_B,Asset_C,Asset_D,Asset_E,Asset_F
Date,,,,,,
2021-03,0.062406,-0.035666,-0.082022,0.081593,0.023528,-0.026178
2021-04,-0.053124,0.010702,-0.046196,0.030448,-0.080412,0.033475
2021-05,-0.081200,0.009674,-0.014500,-0.007936,0.165855,0.082832
2021-06,-0.079557,0.001367,-0.063699,-0.014938,0.190344,0.049044
2021-07,-0.034669,0.140477,0.016832,-0.036071,-0.018636,-0.013185


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np

momentum_monthly = momentum_on_first_days.groupby(momentum_on_first_days.index).first()
momentum_monthly.index = momentum_monthly.index.astype(str)

top2_mask = momentum_monthly.apply(
    lambda row: row.nlargest(2).index.tolist(),
    axis=1
)

fig = go.Figure()

# Heatmap layer
fig.add_trace(
    go.Heatmap(
        z=momentum_monthly.T.values,
        x=momentum_monthly.index,
        y=momentum_monthly.columns,
        colorscale="RdYlGn",
        colorbar=dict(title="30D Momentum")
    )
)

# Overlay markers for Top-2
for month_idx, month in enumerate(momentum_monthly.index):
    for asset in top2_mask.loc[month]:
        asset_idx = list(momentum_monthly.columns).index(asset)

        fig.add_trace(
            go.Scatter(
                x=[month],
                y=[asset],
                mode="markers",
                marker=dict(
                    symbol="star",
                    size=5,
                    color="black"
                ),
                showlegend=False
            )
        )

fig.update_layout(
    title="Monthly Momentum Heatmap (Top-2 Highlighted)",
    xaxis_title="Month",
    yaxis_title="Asset",
    template="plotly_white"
)

fig.show()

In [39]:
top_assets_that_month = (
    momentum_on_first_days
        .apply(lambda row: row.nlargest(2).index.tolist(), axis=1)
        .to_dict()
)
top_assets_that_month

{Period('2021-03', 'M'): ['Asset_D', 'Asset_A'],
 Period('2021-04', 'M'): ['Asset_F', 'Asset_D'],
 Period('2021-05', 'M'): ['Asset_E', 'Asset_F'],
 Period('2021-06', 'M'): ['Asset_E', 'Asset_F'],
 Period('2021-07', 'M'): ['Asset_B', 'Asset_C'],
 Period('2021-08', 'M'): ['Asset_F', 'Asset_D'],
 Period('2021-09', 'M'): ['Asset_B', 'Asset_F'],
 Period('2021-10', 'M'): ['Asset_A', 'Asset_E'],
 Period('2021-11', 'M'): ['Asset_F', 'Asset_C'],
 Period('2021-12', 'M'): ['Asset_D', 'Asset_F'],
 Period('2022-01', 'M'): ['Asset_D', 'Asset_A'],
 Period('2022-02', 'M'): ['Asset_B', 'Asset_C'],
 Period('2022-03', 'M'): ['Asset_E', 'Asset_F'],
 Period('2022-04', 'M'): ['Asset_E', 'Asset_B'],
 Period('2022-05', 'M'): ['Asset_E', 'Asset_F'],
 Period('2022-06', 'M'): ['Asset_A', 'Asset_C'],
 Period('2022-07', 'M'): ['Asset_C', 'Asset_D'],
 Period('2022-08', 'M'): ['Asset_E', 'Asset_D'],
 Period('2022-09', 'M'): ['Asset_A', 'Asset_F'],
 Period('2022-10', 'M'): ['Asset_B', 'Asset_A'],
 Period('2022-11', '

In [40]:
def compute_portfolio_daily_return(top_assets_that_month, daily_returns):

    monthly_period = daily_returns.index.to_period("M")

    portfolio_daily_returns = pd.Series(index=daily_returns.index, dtype=float)

    for month, assets in top_assets_that_month.items():
        mask = monthly_period == month

        portfolio_daily_returns.loc[mask] = (
            daily_returns.loc[mask, assets].mean(axis=1)
        )

    portfolio_daily_returns = portfolio_daily_returns.fillna(0)

    return portfolio_daily_returns

def compute_cumulative_value(portfolio_daily_returns, initial_amount):

    portfolio_value = (
        (1 + portfolio_daily_returns)
        .cumprod()
        * initial_amount
    )

    return portfolio_value

In [41]:
portfolio_daily_returns = compute_portfolio_daily_return(
    top_assets_that_month,
    daily_returns
)

portfolio_value = compute_cumulative_value(
    portfolio_daily_returns,
    initial_amount=1000
)

In [42]:
portfolio_value.index

DatetimeIndex(['2021-01-04', '2021-01-05', '2021-01-06', '2021-01-07',
               '2021-01-08', '2021-01-11', '2021-01-12', '2021-01-13',
               '2021-01-14', '2021-01-18',
               ...
               '2023-12-18', '2023-12-19', '2023-12-20', '2023-12-21',
               '2023-12-22', '2023-12-25', '2023-12-26', '2023-12-27',
               '2023-12-28', '2023-12-29'],
              dtype='datetime64[us]', name='Date', length=740, freq=None)

In [43]:
def total_return(portfolio_value):
  return (portfolio_value.iloc[-1] / portfolio_value.iloc[0]) - 1

In [44]:
total_return(portfolio_value)

np.float64(-0.48717609817008023)

In [45]:
def cagr(portfolio_value, trading_days=252):
    start_value = portfolio_value.iloc[0]
    end_value = portfolio_value.iloc[-1]
    n_days = len(portfolio_value)

    years = n_days / trading_days

    return (end_value / start_value) ** (1 / years) - 1

In [46]:
cagr(portfolio_value)

np.float64(-0.20341443172264306)

In [47]:
def max_drawdown(portfolio_value):
    if portfolio_value.empty:
        return None

    rolling_max = portfolio_value.cummax()
    drawdown = (portfolio_value - rolling_max) / rolling_max

    return drawdown.min()

In [48]:
max_drawdown(portfolio_value)

np.float64(-0.5593899774890101)

In [49]:
def volatility(portfolio_daily_returns, trading_days=252):
    if portfolio_daily_returns.empty:
        return None

    daily_vol = portfolio_daily_returns.std()
    return daily_vol * np.sqrt(trading_days)

In [50]:
volatility(portfolio_daily_returns)

np.float64(0.2228045725705283)

In [51]:
strategy_30_days = {
  "total_return": total_return(portfolio_value),
  "cagr": cagr(portfolio_value),
  "max_drawdown": max_drawdown(portfolio_value),
  "volatility": volatility(portfolio_daily_returns)
}

In [52]:
strategy_30_days

{'total_return': np.float64(-0.48717609817008023),
 'cagr': np.float64(-0.20341443172264306),
 'max_drawdown': np.float64(-0.5593899774890101),
 'volatility': np.float64(0.2228045725705283)}

In [53]:
momentum_score_90_days = get_momentum_score(90, cleaned_data)

In [55]:
momentum_on_first_days = momentum_score_90_days.groupby(momentum_score_90_days.index.to_period("M")).first()
momentum_on_first_days = momentum_on_first_days[1:]

In [56]:
momentum_on_first_days.head()

,Asset_A,Asset_B,Asset_C,Asset_D,Asset_E,Asset_F
Date,,,,,,
2021-06,-0.088161,0.042008,-0.142358,0.040008,0.241979,0.158314
2021-07,-0.160090,0.213376,-0.090054,-0.131122,0.072762,0.079353
2021-08,-0.098766,0.157715,-0.146609,-0.041060,0.190834,0.147174
2021-09,-0.081789,0.214102,-0.106608,-0.130113,0.122316,0.085821
2021-10,0.061335,0.185158,-0.081327,-0.054048,-0.053836,-0.002262


In [57]:
top_assets_that_month = (
    momentum_on_first_days
        .apply(lambda row: row.nlargest(2).index.tolist(), axis=1)
        .to_dict()
)
top_assets_that_month

{Period('2021-06', 'M'): ['Asset_E', 'Asset_F'],
 Period('2021-07', 'M'): ['Asset_B', 'Asset_F'],
 Period('2021-08', 'M'): ['Asset_E', 'Asset_B'],
 Period('2021-09', 'M'): ['Asset_B', 'Asset_E'],
 Period('2021-10', 'M'): ['Asset_B', 'Asset_A'],
 Period('2021-11', 'M'): ['Asset_F', 'Asset_A'],
 Period('2021-12', 'M'): ['Asset_F', 'Asset_B'],
 Period('2022-01', 'M'): ['Asset_C', 'Asset_D'],
 Period('2022-02', 'M'): ['Asset_C', 'Asset_F'],
 Period('2022-03', 'M'): ['Asset_C', 'Asset_F'],
 Period('2022-04', 'M'): ['Asset_F', 'Asset_E'],
 Period('2022-05', 'M'): ['Asset_F', 'Asset_B'],
 Period('2022-06', 'M'): ['Asset_F', 'Asset_E'],
 Period('2022-07', 'M'): ['Asset_E', 'Asset_B'],
 Period('2022-08', 'M'): ['Asset_E', 'Asset_C'],
 Period('2022-09', 'M'): ['Asset_A', 'Asset_D'],
 Period('2022-10', 'M'): ['Asset_A', 'Asset_E'],
 Period('2022-11', 'M'): ['Asset_A', 'Asset_D'],
 Period('2022-12', 'M'): ['Asset_D', 'Asset_A'],
 Period('2023-01', 'M'): ['Asset_D', 'Asset_E'],
 Period('2023-02', '

In [58]:
portfolio_daily_returns = compute_portfolio_daily_return(
    top_assets_that_month,
    daily_returns
)

portfolio_value = compute_cumulative_value(
    portfolio_daily_returns,
    initial_amount=1000
)

In [59]:
strategy_90_days = {
  "total_return": total_return(portfolio_value),
  "cagr": cagr(portfolio_value),
  "max_drawdown": max_drawdown(portfolio_value),
  "volatility": volatility(portfolio_daily_returns)
}

In [60]:
strategy_90_days

{'total_return': np.float64(-0.17937033457895601),
 'cagr': np.float64(-0.06510326013062973),
 'max_drawdown': np.float64(-0.258579493040862),
 'volatility': np.float64(0.2178202347582329)}